In [1]:
# ============================================================
# Notebook    : 14a_case_b_mlp.ipynb
# Description : Case B — MLP contrast architecture (§8.5).
#               Builds B1-MLP and B2-MLP using the EXACT SAME
#               train/val/test split as 06a/06b/13 (single split
#               on the superset feature columns, seed=42), so
#               results are directly comparable to the LightGBM
#               versions without re-deriving alignment guarantees.
#
#               Purpose: defend the §4.6 decision to use LightGBM
#               only — show that a non-tree architecture reaches
#               the same qualitative conclusion (B1->B2 gain is
#               small / not clearly significant) rather than
#               assuming it from Case A's result alone.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install torch scikit-learn pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")


# ============================================================
# 2. Load and split — IDENTICAL to notebook 13's Case B block
#    (single split on the superset feature columns, then B1/B2
#    subset from it), guaranteeing the same test rows as the
#    LightGBM B1/B2 and the significance test in notebook 13.
# ============================================================
df_b = pd.read_csv("data/car_insurance_preprocessed.csv")

NUMERIC_COLS_B1 = ["CREDIT_SCORE", "ANNUAL_MILEAGE", "CHILDREN", "MARRIED", "VEHICLE_OWNERSHIP"]
CAT_COLS_B = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
              "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"]
FLAG_COLS_B = ["CREDIT_SCORE_was_missing", "ANNUAL_MILEAGE_was_missing"]
BEHAV_COLS_B = ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]

FEATURE_COLS_B1 = NUMERIC_COLS_B1 + CAT_COLS_B + FLAG_COLS_B
FEATURE_COLS_B2 = FEATURE_COLS_B1 + BEHAV_COLS_B

X_b_full = df_b[FEATURE_COLS_B2].copy()
y_b_full = df_b["OUTCOME"].copy()

X_train_b, X_temp_b, y_train_b, y_temp_b = train_test_split(
    X_b_full, y_b_full, test_size=0.30, random_state=SEED, stratify=y_b_full
)
X_val_b, X_test_b, y_val_b, y_test_b = train_test_split(
    X_temp_b, y_temp_b, test_size=0.50, random_state=SEED, stratify=y_temp_b
)

print(f"[CHECK 1] Split sizes — train: {len(X_train_b):,}, val: {len(X_val_b):,}, test: {len(X_test_b):,}")
print(f"[CHECK 1] Same split as notebook 13's Case B block (same seed, same superset columns)")


# ============================================================
# 3. Preprocessing for MLP — one-hot encode categoricals,
#    standardize numerics. Fit ONLY on train, applied to val/test
#    (standard leakage-prevention discipline).
# ============================================================
def build_mlp_features(X_train, X_val, X_test, numeric_cols, cat_cols, binary_cols):
    # one-hot: fit categories on train, align val/test to same columns
    X_train_oh = pd.get_dummies(X_train[cat_cols], columns=cat_cols)
    X_val_oh   = pd.get_dummies(X_val[cat_cols],   columns=cat_cols)
    X_test_oh  = pd.get_dummies(X_test[cat_cols],  columns=cat_cols)

    X_val_oh  = X_val_oh.reindex(columns=X_train_oh.columns, fill_value=0)
    X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[numeric_cols + binary_cols])
    X_val_num   = scaler.transform(X_val[numeric_cols + binary_cols])
    X_test_num  = scaler.transform(X_test[numeric_cols + binary_cols])

    X_train_final = np.concatenate([X_train_num, X_train_oh.values.astype(np.float32)], axis=1)
    X_val_final   = np.concatenate([X_val_num,   X_val_oh.values.astype(np.float32)],   axis=1)
    X_test_final  = np.concatenate([X_test_num,  X_test_oh.values.astype(np.float32)],  axis=1)

    return X_train_final, X_val_final, X_test_final


# ============================================================
# 4. MLP model — 2 hidden layers, similar parameter count
#    order-of-magnitude to Case A's Transformer (~88K params)
# ============================================================
class MLP(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_train, y_train, X_val, y_val, n_epochs=50, lr=1e-3, patience=10):
    input_dim = X_train.shape[1]
    model = MLP(input_dim).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())

    pos_rate = y_train.mean()
    pos_weight = torch.tensor((1 - pos_rate) / pos_rate)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t   = torch.tensor(X_val, dtype=torch.float32)
    y_val_t   = torch.tensor(y_val.values, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

    best_val_auc = -1
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits).numpy()
            val_auc = roc_auc_score(y_val_t.numpy(), val_probs)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d} | val_auc: {val_auc:.4f} | best: {best_val_auc:.4f}")

        if epochs_no_improve >= patience:
            print(f"  early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

    model.load_state_dict(best_state)
    return model, n_params, best_val_auc


# ============================================================
# 5. Train B1-MLP (demographics only)
# ============================================================
print("\n" + "="*60)
print("B1-MLP — Demographics only")
print("="*60)

X_train_b1, X_val_b1, X_test_b1 = build_mlp_features(
    X_train_b, X_val_b, X_test_b, NUMERIC_COLS_B1, CAT_COLS_B, FLAG_COLS_B
)
print(f"[CHECK B1] Feature dim after encoding: {X_train_b1.shape[1]}")

model_b1_mlp, n_params_b1, best_val_auc_b1 = train_mlp(
    X_train_b1, y_train_b, X_val_b1, y_val_b
)
print(f"[CHECK B1] Params: {n_params_b1:,}")

model_b1_mlp.eval()
with torch.no_grad():
    test_logits_b1 = model_b1_mlp(torch.tensor(X_test_b1, dtype=torch.float32))
    prob_b1_mlp = torch.sigmoid(test_logits_b1).numpy()

auc_b1_mlp = roc_auc_score(y_test_b.values, prob_b1_mlp)
print(f"[CHECK B1] Test AUC (B1-MLP): {auc_b1_mlp:.4f}")


# ============================================================
# 6. Train B2-MLP (demographics + behavioral)
# ============================================================
print("\n" + "="*60)
print("B2-MLP — Demographics + behavioral")
print("="*60)

X_train_b2, X_val_b2, X_test_b2 = build_mlp_features(
    X_train_b, X_val_b, X_test_b, NUMERIC_COLS_B1, CAT_COLS_B, FLAG_COLS_B + BEHAV_COLS_B
)
print(f"[CHECK B2] Feature dim after encoding: {X_train_b2.shape[1]}")

model_b2_mlp, n_params_b2, best_val_auc_b2 = train_mlp(
    X_train_b2, y_train_b, X_val_b2, y_val_b
)
print(f"[CHECK B2] Params: {n_params_b2:,}")

model_b2_mlp.eval()
with torch.no_grad():
    test_logits_b2 = model_b2_mlp(torch.tensor(X_test_b2, dtype=torch.float32))
    prob_b2_mlp = torch.sigmoid(test_logits_b2).numpy()

auc_b2_mlp = roc_auc_score(y_test_b.values, prob_b2_mlp)
print(f"[CHECK B2] Test AUC (B2-MLP): {auc_b2_mlp:.4f}")


# ============================================================
# 7. Paired bootstrap — same function as notebook 13, reused
#    here to test B1-MLP vs B2-MLP significance
# ============================================================
def paired_bootstrap_auc_diff(y_true, prob_1, prob_2, n_boot=1000, seed=42, alpha=0.05):
    y_true = np.asarray(y_true)
    n = len(y_true)
    point_diff = roc_auc_score(y_true, prob_2) - roc_auc_score(y_true, prob_1)
    rng = np.random.RandomState(seed)
    diffs = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        yt_b = y_true[idx]
        if len(np.unique(yt_b)) < 2:
            diffs[b] = np.nan
            continue
        diffs[b] = roc_auc_score(yt_b, prob_2[idx]) - roc_auc_score(yt_b, prob_1[idx])
    diffs = diffs[~np.isnan(diffs)]
    ci_low, ci_high = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    p_approx = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return point_diff, ci_low, ci_high, not (ci_low <= 0 <= ci_high), p_approx

diff, ci_low, ci_high, sig, p = paired_bootstrap_auc_diff(y_test_b.values, prob_b1_mlp, prob_b2_mlp)
print(f"\n[SIGNIFICANCE] B1-MLP vs B2-MLP: diff={diff:+.4f}, "
      f"95% CI=[{ci_low:+.4f}, {ci_high:+.4f}], "
      f"significant={sig}, p_approx={p:.4f}")


# ============================================================
# 8. Save models and predictions
# ============================================================
torch.save(model_b1_mlp.state_dict(), "data/sequences/case_b_mlp_demographic_model.pt")
torch.save(model_b2_mlp.state_dict(), "data/sequences/case_b_mlp_full_model.pt")
np.savez("data/sequences/case_b_mlp_test_predictions.npz",
         labels=y_test_b.values, prob_b1_mlp=prob_b1_mlp, prob_b2_mlp=prob_b2_mlp)
print("\nSaved MLP models and predictions.")


# ============================================================
# 9. Comparison table — MLP vs LightGBM, side by side
# ============================================================
print("\n===== Case B: Architecture Comparison Summary =====")
comparison = pd.DataFrame([
    {"Feature_Set": "Demographics only", "LightGBM_AUC": 0.8875, "MLP_AUC": round(auc_b1_mlp, 4)},
    {"Feature_Set": "Demographics+Behavioral", "LightGBM_AUC": 0.8894, "MLP_AUC": round(auc_b2_mlp, 4)},
])
print(comparison.to_string(index=False))
print(f"\nLightGBM improvement (B1->B2): +0.0018 (not significant, p=0.366, from notebook 13)")
print(f"MLP improvement (B1-MLP->B2-MLP): {diff:+.4f} (significant={sig}, p_approx={p:.4f})")

comparison.to_csv("paper_assets/tables/Table4_CaseB_Architecture_Comparison.csv", index=False)
print("\nSaved: paper_assets/tables/Table4_CaseB_Architecture_Comparison.csv")

[CHECK 1] Split sizes — train: 7,000, val: 1,500, test: 1,500
[CHECK 1] Same split as notebook 13's Case B block (same seed, same superset columns)

B1-MLP — Demographics only
[CHECK B1] Feature dim after encoding: 30
  epoch   1 | val_auc: 0.8685 | best: 0.8685
  epoch  10 | val_auc: 0.8885 | best: 0.8892
  early stopping at epoch 16 (no improvement for 10 epochs)
[CHECK B1] Params: 4,097
[CHECK B1] Test AUC (B1-MLP): 0.8886

B2-MLP — Demographics + behavioral
[CHECK B2] Feature dim after encoding: 33
  epoch   1 | val_auc: 0.8658 | best: 0.8658
  epoch  10 | val_auc: 0.8896 | best: 0.8896
  early stopping at epoch 18 (no improvement for 10 epochs)
[CHECK B2] Params: 4,289
[CHECK B2] Test AUC (B2-MLP): 0.8878

[SIGNIFICANCE] B1-MLP vs B2-MLP: diff=-0.0008, 95% CI=[-0.0046, +0.0029], significant=False, p_approx=0.7220

Saved MLP models and predictions.

===== Case B: Architecture Comparison Summary =====
            Feature_Set  LightGBM_AUC  MLP_AUC
      Demographics only        0.88